# MeteoSwiss OGD — Solar Production Estimate

This notebook shows a practical use case for the [MeteoSwiss Open Government Data (OGD)](https://opendatadocs.meteoswiss.ch/e-forecast-data/e4-local-forecast-data) local forecast: estimating the expected output of a photovoltaic (PV) system from the 9-day radiation forecast.

By the end you will know how to:
1. Fetch the same local-forecast data used in `Meteogram.ipynb` (global radiation, cloud cover)
2. Feed it into a simple physical PV model
3. Plot the estimated hourly power and daily energy yield

**Note:** the PV model here is deliberately simple and illustrative — it assumes an idealised, unshaded, south-facing panel and a fixed system derate factor. It is meant to demonstrate how OGD forecast data can feed a downstream model, not to be a calibrated PV yield tool. For production-grade PV forecasting, consider a package such as [`pvlib`](https://pvlib-python.readthedocs.io/), which accounts for panel tilt/orientation, solar position, and temperature effects.

**Data source:** `ch.meteoschweiz.ogd-local-forecasting` — updated hourly  
**STAC API:** https://data.geo.admin.ch/api/stac/v1

## 0 · Setup

Same OGD endpoints and parameter metadata as `Meteogram.ipynb` — see that notebook for a
step-by-step explanation of the API. Here we only fetch what the PV model needs:
global radiation (`gre000h0`) and cloud cover, for context.

In [ ]:
import httpx
import pandas as pd
from io import StringIO
from datetime import datetime
from zoneinfo import ZoneInfo

# Plotting module — lives alongside this notebook
from solar_plot import plot_solar_production

# ---------------------------------------------------------------------------
# API endpoints
# ---------------------------------------------------------------------------
STAC_BASE_URL  = "https://data.geo.admin.ch/api/stac/v1"
COLLECTION_ID  = "ch.meteoschweiz.ogd-local-forecasting"
METADATA_URL   = (
    f"https://data.geo.admin.ch/{COLLECTION_ID}/"
    "ogd-local-forecasting_meta_parameters.csv"
)
POI_LIST_URL   = (
    f"https://data.geo.admin.ch/{COLLECTION_ID}/"
    "ogd-local-forecasting_meta_point.csv"
)

LOCAL_TZ = ZoneInfo("Europe/Zurich")

# ---------------------------------------------------------------------------
# Load parameter metadata
# ---------------------------------------------------------------------------
resp = httpx.get(METADATA_URL, follow_redirects=True, timeout=30)
resp.raise_for_status()
meta_df = pd.read_csv(StringIO(resp.content.decode("latin-1")), sep=";")

PARAM_UNITS  = dict(zip(meta_df["parameter_shortname"], meta_df["parameter_unit"]))
DAILY_PARAMS = set(meta_df.loc[meta_df["parameter_granularity"] == "D", "parameter_shortname"])

print(f"✓ {len(meta_df)} parameters loaded from OGD metadata")

## 1 · Configuration

Choose a location (same `point_id` / `point_type_id` scheme as `Meteogram.ipynb` — use that
notebook's searchable POI table to find one) and describe the illustrative PV system.

In [ ]:
# ── User configuration ────────────────────────────────────────────────────
POI_ID      = "118802"   # point_id from Meteogram.ipynb's POI table
POI_TYPE_ID = "2"        # point_type_id from the same table

PV_CAPACITY_KWP = 5.0    # illustrative rooftop system size (kWp)
DERATE_FACTOR   = 0.85   # illustrative system losses: inverter, wiring, soiling, temperature
# ─────────────────────────────────────────────────────────────────────────

PARAMS_NEEDED = ["gre000h0", "nprolohs", "npromths", "nprohihs"]

print(f"POI {POI_ID}/{POI_TYPE_ID} | PV system: {PV_CAPACITY_KWP} kWp, derate {DERATE_FACTOR}")

## 2 · Download the forecast data

Same three steps as `Meteogram.ipynb`: resolve today's STAC item, download the parameter
CSVs we need, then filter each to our point of interest.

In [ ]:
# Step 1 — Resolve the POI and fetch today's STAC item
df_pois = pd.read_csv(POI_LIST_URL, sep=";", encoding="latin-1")

poi_match = df_pois[
    (df_pois["point_id"] == int(POI_ID)) &
    (df_pois["point_type_id"] == int(POI_TYPE_ID))
]
if poi_match.empty:
    raise ValueError(f"Location with point_id {POI_ID} and point_type_id {POI_TYPE_ID} not found.")

poi_row  = poi_match.iloc[0]
poi_name = poi_row["point_name"]
if "point_height_masl" in poi_row:
    elev_str = f", {poi_row['point_height_masl']:.0f} m a.s.l."
else:
    elev_str = ""

today_id = f"{datetime.now(LOCAL_TZ).strftime('%Y%m%d')}-ch"
item_url = f"{STAC_BASE_URL}/collections/{COLLECTION_ID}/items/{today_id}"

with httpx.Client() as client:
    item = client.get(item_url)
    item.raise_for_status()
    stac_item = item.json()

assets = stac_item["assets"]
# Asset key format: "vnut12.lssw.202607030800.gre000h0.csv" — the 3rd field is the run timestamp
all_runs   = sorted({key.split(".")[2] for key in assets})
latest_run = all_runs[-1]
runtime_dt = (
    datetime.strptime(latest_run, "%Y%m%d%H%M")
    .replace(tzinfo=ZoneInfo("UTC"))
    .astimezone(LOCAL_TZ)
)

param_urls = {}
for param in PARAMS_NEEDED:
    for key, asset in assets.items():
        if param in key:
            param_urls[param] = asset["href"]
            break

print(f"✓ Selected POI: {POI_ID} (Type {POI_TYPE_ID}) — {poi_name}{elev_str}")
print(f"  Latest run: {runtime_dt.strftime('%Y-%m-%d %H:%M %Z')}")
print(f"  {len(param_urls)}/{len(PARAMS_NEEDED)} parameters matched to asset URLs")

In [ ]:
# Step 2 — Download and parse each parameter CSV, filtered to our POI
def parse_parameter_csv(content: bytes, param: str, poi_id: str, poi_type_id: str, tz) -> pd.DataFrame:
    """Parse a single parameter CSV and return a time-indexed Series for the unique POI."""
    df = pd.read_csv(StringIO(content.decode("latin-1")), sep=";")
    df = df[
        (df["point_id"].astype(str) == poi_id) &
        (df["point_type_id"].astype(str) == poi_type_id)
    ]
    if df.empty:
        return pd.DataFrame()
    time_col = next(c for c in df.columns if c.lower() in ("date", "time"))
    df[time_col] = (
        pd.to_datetime(df[time_col].astype(int).astype(str), format="%Y%m%d%H%M", utc=True)
        .dt.tz_convert(tz)
    )
    series = df.set_index(time_col)[[param]]
    series[param] = pd.to_numeric(series[param], errors="coerce")
    return series

poi_id      = str(poi_row["point_id"])
poi_type_id = str(poi_row["point_type_id"])
dfs_hourly  = {}

with httpx.Client(timeout=30.0) as client:
    for param, url in param_urls.items():
        resp = client.get(url)
        resp.raise_for_status()
        series = parse_parameter_csv(resp.content, param, poi_id, poi_type_id, LOCAL_TZ)
        if series.empty:
            print(f"  ⚠ No data for POI {poi_id}/{poi_type_id} in {param}")
            continue
        dfs_hourly[param] = series

df_hourly = pd.concat(dfs_hourly.values(), axis=1).sort_index() if dfs_hourly else pd.DataFrame()

print(f"✓ df_hourly: {df_hourly.shape[0]} timesteps × {df_hourly.shape[1]} parameters")
print(f"  Range: {df_hourly.index[0].strftime('%Y-%m-%d %H:%M')} → {df_hourly.index[-1].strftime('%Y-%m-%d %H:%M')}")
display(df_hourly.head(3))

## 3 · A simple physical PV model

We estimate instantaneous PV power as proportional to forecasted global radiation
(`gre000h0`, W/m²) relative to the standard test condition irradiance of 1000 W/m²,
scaled by the system's capacity and a fixed derate factor:

```
P(t) = capacity_kwp × (gre000h0(t) / 1000) × derate_factor
```

**Assumptions and limitations** (this is intentionally a back-of-envelope model):
- No panel tilt, orientation, or solar-position geometry — radiation is used as-is
- No temperature derating of panel efficiency
- No shading, snow, or soiling beyond the flat `DERATE_FACTOR`
- Cloud cover columns are fetched for context/plotting only, not used in the formula
  (they are already reflected in the `gre000h0` forecast itself)

Daily energy is the sum of hourly power over each day (kWh = ∑ kW × 1 h).

In [ ]:
STC_IRRADIANCE = 1000  # W/m², standard test condition irradiance used to normalise radiation

df_hourly["pv_power_kw"] = (
    PV_CAPACITY_KWP * (df_hourly["gre000h0"] / STC_IRRADIANCE) * DERATE_FACTOR
).clip(lower=0, upper=PV_CAPACITY_KWP)

daily_power = df_hourly["pv_power_kw"].groupby(df_hourly.index.date)
df_daily = daily_power.sum().to_frame(name="pv_energy_kwh")
df_daily.index = pd.to_datetime(df_daily.index)

# Drop partial days at the edges of the forecast window
df_daily = df_daily[daily_power.count() >= 20]

print(f"✓ Estimated peak hourly power: {df_hourly['pv_power_kw'].max():.2f} kW")
print(f"✓ Estimated daily energy range: {df_daily['pv_energy_kwh'].min():.1f}–{df_daily['pv_energy_kwh'].max():.1f} kWh")
display(df_daily)

## 4 · Visualise — estimated PV production

The `plot_solar_production()` function assembles the hourly power and daily energy chart.
All matplotlib code lives in `solar_plot.py` — open it if you want to customise colours or layout.

In [ ]:
plot_solar_production(
    df_hourly       = df_hourly,
    df_daily        = df_daily,
    poi_name        = poi_name,
    elev_str        = elev_str,
    runtime_dt      = runtime_dt,
    local_tz        = LOCAL_TZ,
    pv_capacity_kwp = PV_CAPACITY_KWP,
)

## Next steps

- **Better PV models:** replace the simple radiation-ratio formula with [`pvlib`](https://pvlib-python.readthedocs.io/) to account for panel tilt/orientation, solar position, and temperature derating.
- **Other locations:** change `POI_ID`/`POI_TYPE_ID` — use `Meteogram.ipynb`'s searchable POI table to find one.
- **Full weather picture:** see `Meteogram.ipynb` for the complete 6-panel forecast (temperature, precipitation, wind, sunshine, radiation, clouds).
- **OGD documentation:** https://meteoswiss.atlassian.net/wiki/spaces/DAT/pages/1734410344/OGD
- **STAC API reference:** https://data.geo.admin.ch/api/stac/v1